# Bronze

 Neste notebook, iremos colocar os dados obtidos via API na camada Bronze. 

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

### Vamos criar um dataframe com spark para que possamos explorar os dados.

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/Volumes/workspace/bronze/dbs/data.csv")

In [0]:
# Vamos visualizar os dados sem qualquer transformação 
df.limit(50).display()

In [0]:
df.printSchema()

In [0]:
# Vamos extrair ano, mês e dia da coluna 'data'
# antes disso, garantimos que a coluna está no tipo correto
df = df.withColumn("data", F.to_date("data"))

df = df.withColumn("ano", F.year("data")) \
       .withColumn("mes", F.month("data")) \
       .withColumn("dia", F.dayofmonth("data"))

# Adicionando a coluna 'data_carga_lake' com a data atual no formato yyyy-MM-dd
# Essa coluna registra quando os dados foram carregados no Data Lake,
# sendo essencial para auditoria, rastreabilidade e controle de versões dos dados.
df = df.withColumn("data_carga_lake", F.current_date())

In [0]:
df.limit(50).display()


## Vamos salvar os dados brutos na camada Bronze



In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
SHOW VOLUMES IN workspace.bronze;

In [0]:
# criar pasta para gravar o arquivo parquet

dbutils.fs.mkdirs("/Volumes/workspace/bronze/dbs/Sinistros_Transito")

#### Gravação dos dados em formato Delta (Parquet)

Nesta etapa, os dados são gravados utilizando o formato **Delta Lake**, que é construído sobre arquivos **Parquet**.

O Parquet é um formato de armazenamento colunar, permitindo maior eficiência na compressão e melhor desempenho em consultas, pois apenas as colunas necessárias são lidas durante o processamento.

O Delta Lake adiciona funcionalidades importantes ao Parquet, como controle de versão, transações ACID e suporte a cargas incrementais. Isso garante maior confiabilidade no processamento de dados, evitando inconsistências e permitindo operações como inserção, atualização e merge de dados.

Além disso, será criada uma **tabela externa no Databricks SQL Warehouse**, apontando para os arquivos armazenados no Data Lake. Dessa forma, os dados podem ser consultados via SQL sem a necessidade de duplicação, facilitando a integração com ferramentas analíticas e de visualização.

Com essa abordagem, obtemos um pipeline mais eficiente, escalável e confiável para o processamento de grandes volumes de dados.

In [0]:

# Grava os dados no formato Delta Lake no Data Lake, sobrescrevendo os existentes e permitindo evolução do schema
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .save("/Volumes/workspace/bronze/dbs/Sinistros_Transito/Sinistros_Transito_open_data")

In [0]:
# Cria o database bronze caso não exista
spark.sql("""CREATE DATABASE IF NOT EXISTS bronze;""")

# Registra a tabela Delta no catálogo a partir dos dados processados
df.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .saveAsTable("workspace.bronze.tb_Sinistros_Transito_open_data")


### Para garantir que os dados foram inseridos na camada Bronze, vamos visualizar 

In [0]:
spark.sql("""
    SELECT * 
    FROM workspace.bronze.tb_Sinistros_Transito_open_data
    LIMIT 15
""").display()